# A/B Test Analysis — Landing Page Conversion
**Dataset:** Kaggle A/B Testing Dataset (`ab_data.csv`)  
**Tool:** DuckDB + Python  
**Goal:** Determine whether the new landing page significantly improves conversion rate

## Step 1 — Setup & Load Data

In [3]:
import duckdb
import pandas as pd

conn = duckdb.connect('ab_test.db')

conn.execute("""
    CREATE TABLE IF NOT EXISTS ab_data AS
    SELECT * FROM read_csv_auto('ab_data.csv', header=true)
""")

# Preview first 5 rows
conn.execute("SELECT * FROM ab_data LIMIT 5").df()

,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


## Step 2 — Data Shape

In [4]:
# Total row count
conn.execute("SELECT COUNT(*) AS total_rows FROM ab_data").df()

,total_rows
0,294478


In [5]:
# Column names and types
conn.execute("PRAGMA table_info('ab_data')").df()

,cid,name,type,notnull,dflt_value,pk
0,0,user_id,BIGINT,False,None,False
1,1,timestamp,TIMESTAMP,False,None,False
2,2,group,VARCHAR,False,None,False
3,3,landing_page,VARCHAR,False,None,False
4,4,converted,BIGINT,False,None,False


In [6]:
# Distinct values and date range
conn.execute("""
    SELECT
        COUNT(DISTINCT user_id)      AS unique_users,
        COUNT(DISTINCT "group")      AS groups,
        COUNT(DISTINCT landing_page) AS page_types,
        MIN(timestamp)               AS earliest,
        MAX(timestamp)               AS latest
    FROM ab_data
""").df()

,unique_users,groups,page_types,earliest,latest
0,290584,2,2,2017-01-02 13:42:05.378582,2017-01-24 13:41:54.460509


In [7]:
# Check group and page label values
conn.execute("""
    SELECT DISTINCT "group", landing_page
    FROM ab_data
    ORDER BY "group", landing_page
""").df()

,group,landing_page
0,control,new_page
1,control,old_page
2,treatment,new_page
3,treatment,old_page


## Step 3 — Data Quality Checks

In [8]:
# Check for NULLs across all columns
conn.execute("""
    SELECT
        SUM(CASE WHEN user_id      IS NULL THEN 1 ELSE 0 END) AS null_user_id,
        SUM(CASE WHEN timestamp    IS NULL THEN 1 ELSE 0 END) AS null_timestamp,
        SUM(CASE WHEN "group"      IS NULL THEN 1 ELSE 0 END) AS null_group,
        SUM(CASE WHEN landing_page IS NULL THEN 1 ELSE 0 END) AS null_landing_page,
        SUM(CASE WHEN converted    IS NULL THEN 1 ELSE 0 END) AS null_converted
    FROM ab_data
""").df()

,null_user_id,null_timestamp,null_group,null_landing_page,null_converted
0,0.0,0.0,0.0,0.0,0.0


In [9]:
# Check for duplicate user IDs
conn.execute("""
    SELECT
        COUNT(*)                           AS total_rows,
        COUNT(DISTINCT user_id)            AS unique_users,
        COUNT(*) - COUNT(DISTINCT user_id) AS duplicate_rows
    FROM ab_data
""").df()

,total_rows,unique_users,duplicate_rows
0,294478,290584,3894


In [10]:
# Check group/page assignment breakdown
# Expect: control=old_page, treatment=new_page only
conn.execute("""
    SELECT "group", landing_page, COUNT(*) AS row_count
    FROM ab_data
    GROUP BY "group", landing_page
    ORDER BY "group", landing_page
""").df()

,group,landing_page,row_count
0,control,new_page,1928
1,control,old_page,145274
2,treatment,new_page,145311
3,treatment,old_page,1965


In [11]:
# Count mismatched rows (users shown the wrong page)
conn.execute("""
    SELECT COUNT(*) AS mismatched_rows
    FROM ab_data
    WHERE ("group" = 'treatment' AND landing_page = 'old_page')
       OR ("group" = 'control'   AND landing_page = 'new_page')
""").df()

,mismatched_rows
0,3893


## Step 4 — Clean Data

In [12]:
# Remove mismatches and duplicate user IDs
conn.execute("""
    CREATE OR REPLACE VIEW ab_clean AS
    SELECT *
    FROM ab_data
    WHERE NOT (
        ("group" = 'treatment' AND landing_page = 'old_page')
        OR
        ("group" = 'control'   AND landing_page = 'new_page')
    )
    QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY timestamp) = 1
""")

conn.execute("SELECT COUNT(*) AS clean_rows FROM ab_clean").df()

,clean_rows
0,290584


## Step 5 — Core Conversion Analysis

In [13]:
# Headline conversion rate by group
conn.execute("""
    SELECT
        "group",
        COUNT(*)                       AS total_users,
        SUM(converted)                 AS conversions,
        ROUND(AVG(converted) * 100, 4) AS conversion_rate_pct
    FROM ab_clean
    GROUP BY "group"
    ORDER BY "group"
""").df()

,group,total_users,conversions,conversion_rate_pct
0,control,145274,17489.0,12.0386
1,treatment,145310,17264.0,11.8808


In [14]:
# Uplift calculation
conn.execute("""
    WITH rates AS (
        SELECT "group", AVG(converted) AS conv_rate
        FROM ab_clean
        GROUP BY "group"
    )
    SELECT
        ROUND(MAX(CASE WHEN "group" = 'control'   THEN conv_rate END) * 100, 4) AS control_pct,
        ROUND(MAX(CASE WHEN "group" = 'treatment' THEN conv_rate END) * 100, 4) AS treatment_pct,
        ROUND(
            (MAX(CASE WHEN "group" = 'treatment' THEN conv_rate END)
           - MAX(CASE WHEN "group" = 'control'   THEN conv_rate END))
           / MAX(CASE WHEN "group" = 'control'   THEN conv_rate END) * 100
        , 2) AS uplift_pct
    FROM rates
""").df()

,control_pct,treatment_pct,uplift_pct
0,12.0386,11.8808,-1.31


In [15]:
# Sample ratio mismatch check — groups should be roughly 50/50
conn.execute("""
    SELECT
        "group",
        COUNT(*) AS n,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
    FROM ab_clean
    GROUP BY "group"
""").df()

,group,n,pct_of_total
0,control,145274,49.99
1,treatment,145310,50.01


## Step 6 — Time-Series Exploration

In [16]:
# Daily conversion rate per group
conn.execute("""
    SELECT
        CAST(timestamp AS DATE)        AS day,
        "group",
        COUNT(*)                       AS users,
        ROUND(AVG(converted) * 100, 2) AS daily_conv_rate_pct
    FROM ab_clean
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df()

,day,group,users,daily_conv_rate_pct
0,2017-01-02,control,2859,12.56
1,2017-01-02,treatment,2853,11.99
2,2017-01-03,control,6590,11.38
3,2017-01-03,treatment,6618,11.38
4,2017-01-04,control,6578,12.19
5,2017-01-04,treatment,6541,11.66
6,2017-01-05,control,6427,12.32
7,2017-01-05,treatment,6505,11.50
8,2017-01-06,control,6606,11.53
9,2017-01-06,treatment,6747,12.35


In [17]:
# Cumulative conversion rate over time
conn.execute("""
    SELECT
        CAST(timestamp AS DATE) AS day,
        "group",
        SUM(SUM(converted)) OVER (PARTITION BY "group" ORDER BY CAST(timestamp AS DATE)) AS cumulative_conversions,
        SUM(COUNT(*))        OVER (PARTITION BY "group" ORDER BY CAST(timestamp AS DATE)) AS cumulative_users,
        ROUND(
            SUM(SUM(converted)) OVER (PARTITION BY "group" ORDER BY CAST(timestamp AS DATE))
            / NULLIF(SUM(COUNT(*)) OVER (PARTITION BY "group" ORDER BY CAST(timestamp AS DATE)), 0)
            * 100, 4
        ) AS cumulative_conv_rate_pct
    FROM ab_clean
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df()

,day,group,cumulative_conversions,cumulative_users,cumulative_conv_rate_pct
0,2017-01-02,control,359.0,2859.0,12.5568
1,2017-01-02,treatment,342.0,2853.0,11.9874
2,2017-01-03,control,1109.0,9449.0,11.7367
3,2017-01-03,treatment,1095.0,9471.0,11.5616
4,2017-01-04,control,1911.0,16027.0,11.9236
5,2017-01-04,treatment,1858.0,16012.0,11.6038
6,2017-01-05,control,2703.0,22454.0,12.0379
7,2017-01-05,treatment,2606.0,22517.0,11.5735
8,2017-01-06,control,3465.0,29060.0,11.9236
9,2017-01-06,treatment,3439.0,29264.0,11.7516


## Step 7 — Export CSVs for Power BI

In [18]:
# Export 1: Summary stats (KPI cards + bar chart)
conn.execute("""
    COPY (
        SELECT
            "group",
            COUNT(*)                       AS total_users,
            SUM(converted)                 AS conversions,
            ROUND(AVG(converted) * 100, 4) AS conversion_rate_pct
        FROM ab_clean
        GROUP BY "group"
    ) TO 'output_summary.csv' (HEADER, DELIMITER ',')
""")
print("output_summary.csv exported")

# Export 2: Daily time series (trend line)
conn.execute("""
    COPY (
        SELECT
            CAST(timestamp AS DATE)        AS day,
            "group",
            COUNT(*)                       AS users,
            ROUND(AVG(converted) * 100, 4) AS daily_conv_rate_pct
        FROM ab_clean
        GROUP BY 1, 2
        ORDER BY 1, 2
    ) TO 'output_daily.csv' (HEADER, DELIMITER ',')
""")
print("output_daily.csv exported")

# Export 3: Full clean dataset
conn.execute("""
    COPY (
        SELECT * FROM ab_clean
    ) TO 'output_clean.csv' (HEADER, DELIMITER ',')
""")
print("output_clean.csv exported")

print("\nAll exports complete — load these 3 files into Power BI")

output_summary.csv exported
output_daily.csv exported
output_clean.csv exported

All exports complete — load these 3 files into Power BI
